### Data Collector Notebook

This notebook contains code to export data from the Yahoo Finance API to .csv files.

In [2]:
import yfinance as yf
from datetime import date, timedelta
from pandas import DataFrame
import numpy as np
import os
import stockdex as sd

DATE_LEN = 10


In [3]:
def save_phist_for_symbols(syms: list[str], start_date: date, end_date: date, interval: str = "1d", file='raw_phist.csv'):
    phistory = yf.download(syms, start=start_date, end=end_date, interval=interval, auto_adjust=True)

    if os.path.exists(file): os.remove(file)

    phistory = phistory.drop(labels=["High", "Low", "Open"], axis=1)

    phistory = phistory.dropna(axis=1, how="all")

    phistory.to_csv(file)

In [4]:
def parse_symbol_list(file: str, max: int = 5000, skip_header: bool = True, exclude_fishy: bool = True) -> list[str]:
    output = []

    def is_fishy(sym: str) -> bool:
        return sym.find('.') != -1

    with open(file) as f:
        if skip_header: f.readline()
        count = 0
        while (line := f.readline()) != '' and count < max:
            sym = line.split('\t')[0]
            if not is_fishy(sym) or not exclude_fishy:
                output.append(sym)
            count += 1

    return output


In [5]:
def save_macro_trends_for_symbols(syms: list[str], start_date: date, end_date: date):
    for sym in syms:
        ticker = sd.Ticker(ticker=sym)
        #income_st = ticker.macrotrends_income_statement(frequency="quarterly")

        #print(income_st)

        #kfrs = ticker.macrotrends_key_financial_ratios.dropna(axis=0, how='any')

        #print(kfrs)

        ebitas = ticker.macrotrends_ebitda_margin.drop(['TTM Revenue', 'TTM EBITDA'], axis=1)

        earnings = ticker.finviz_earnings_data().drop(['fiscalPeriod', 'earningsDate', 'salesActual', 'epsReportedActual', 'epsReportedEstimate', 'salesEstimate', 'epsReportedAnalysts', 'salesAnalysts'], axis=1)

        margin = ticker.macrotrends_operating_margin.drop(['TTM Revenue', 'TTM Operating Income'], axis=1).dropna(axis=0, how='all')

        #print(ebitas)

        print(earnings)

        #print(margin)

In [6]:
save_macro_trends_for_symbols(['AAPL'], None, None)

    ticker fiscalEndDate  epsActual  epsEstimate  epsAnalysts
0     AAPL    2025-12-31       2.84       2.6733         31.0
1     AAPL    2025-09-30       1.85       1.7771         31.0
2     AAPL    2025-06-30       1.57       1.4380         29.0
3     AAPL    2025-03-31       1.65       1.6273         31.0
4     AAPL    2024-12-31       2.40       2.3477         32.0
..     ...           ...        ...          ...          ...
111   AAPL    2027-09-30        NaN       2.1954         18.0
112   AAPL    2027-12-31        NaN       3.2058          7.0
113   AAPL    2028-03-31        NaN       2.3455          4.0
114   AAPL    2028-06-30        NaN       2.1235          4.0
115   AAPL    2028-09-30        NaN       2.4048          4.0

[116 rows x 5 columns]


In [6]:
# Test Save is working on a single ticker
#save_phist_for_symbol("AAPL", date(2000, 1, 1), date(2026,1,1))

In [ ]:
symbols = parse_symbol_list("Symbols_TSX.txt")
print("Loading", len(symbols), "symbols...")

Loading 56 symbols...


In [7]:
symbols = []

with open('s&p500.csv') as file:
    count = 0
    while (line := file.readline()) != '' and count < 510:
        sym = line.split(',')[0]
        symbols.append(sym)
        count += 1

len(symbols)

504

In [8]:
save_phist_for_symbols(symbols, date(1980, 1, 1), date(2026, 4, 1))

[                       1%                       ]  4 of 504 completed$BRK.B: possibly delisted; no timezone found
[*****                 11%                       ]  57 of 504 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SYMBOL"}}}
[******                12%                       ]  61 of 504 completed$SYMBOL: possibly delisted; no timezone found
[**********************65%******                 ]  327 of 504 completed$BF.B: possibly delisted; no price data found  (1d 1980-01-01 -> 2026-04-01)
[*********************100%***********************]  504 of 504 completed

3 Failed downloads:
['BRK.B', 'SYMBOL']: possibly delisted; no timezone found
['BF.B']: possibly delisted; no price data found  (1d 1980-01-01 -> 2026-04-01)


In [2]:
SP500 = "^GSPC"

phistory = yf.download(SP500, start=date(1980, 1, 1), end=date(2026, 4, 1), interval="1d", auto_adjust=True)

if os.path.exists("s&p_phist.csv"): os.remove("s&p_phist.csv")

phistory = phistory.drop(labels=["High", "Low", "Open"], axis=1)

phistory = phistory.dropna(axis=1, how="all")

phistory.to_csv("s&p_phist.csv")

[*********************100%***********************]  1 of 1 completed


In [64]:
symbols = []

with open("TSX_60.txt") as f:
    while (line := f.readline()) != '':
        sym = line.split('-')[0]
        symbols.append(sym)

save_phist_for_symbols(symbols, date(1980, 1, 1), date(2026, 4, 1), file='raw_tsx60.csv')

$ATD: possibly delisted; no price data found  (1d 1980-01-01 -> 2026-04-01)
[***********           23%                       ]  13 of 57 completed$TOU: possibly delisted; no price data found  (1d 1980-01-01 -> 2026-04-01)
[**************        30%                       ]  17 of 57 completed$WSP: possibly delisted; no price data found  (1d 1980-01-01 -> 2026-04-01)
[******************    37%                       ]  21 of 57 completed$K: possibly delisted; no timezone found
[**********************53%                       ]  30 of 57 completed$FFH: possibly delisted; no price data found  (1d 1980-01-01 -> 2026-04-01)
[**********************67%*******                ]  38 of 57 completed$CSU: possibly delisted; no timezone found
[*********************100%***********************]  57 of 57 completed

6 Failed downloads:
['ATD', 'TOU', 'WSP', 'FFH']: possibly delisted; no price data found  (1d 1980-01-01 -> 2026-04-01)
['K', 'CSU']: possibly delisted; no timezone found
